# Long-Context TTFT Crossover Sweep

Companion to the T4 sweep in the paper's break-even section. That sweep used
TinyLlama-1.1B, whose 2,048-token context window capped it at 1,792 tokens, only
just past its predicted crossover of N* ~ 1,561. It could show the curve crossing
parity but not what happens well beyond it.

This notebook repeats the measurement on a long-context model so the sweep spans a
decade of context length entirely above N*, turning 'brackets the prediction' into
'tracks the prediction'.

**Runtime:** GPU (a free T4 suffices). **Expected time:** 15-25 minutes.

**Output:** `context_sweep_results.json` -- download it and hand it back.


In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install 'transformers>=4.44,<5' accelerate 2>&1 | tail -1


Tesla T4, 15360 MiB
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## Configuration

`Qwen2.5-0.5B` is the default because it is ungated (no HuggingFace token required)
and has a 32k context window. `Llama-3.2-1B` is a closer architectural match to the
paper's GQA discussion but is gated; select it only if this account has accepted its
license.


In [2]:
MODEL = 'meta-llama/Llama-3.2-1B'    # gated: see the authentication cell below
# MODEL = 'Qwen/Qwen2.5-0.5B'        # ungated fallback, 32k context

CONTEXT_LENGTHS = [512, 2048, 4096, 8192, 16384]
TRIALS = 5      # per length; medians are reported

# A T4 is compute capability 7.5. PyTorch's *flash* SDPA backend needs sm80+,
# so on a T4 attention falls back to the memory-efficient or math backend. The
# math backend materializes the full N x N attention matrix, which OOMs a 16 GB
# card at 16k tokens and inflates prefill time. We pin a tiled backend below.
FORCE_MEM_EFFICIENT_ATTENTION = True


## HuggingFace authentication

`meta-llama/Llama-3.2-1B` is a gated repository. Two one-time steps:

1. Accept the license at
   [huggingface.co/meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B)
   while signed in. Approval is usually immediate.
2. Create a **read** token at
   [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

Then give Colab the token. The tidier way is the key icon in the left sidebar:
add a secret named `HF_TOKEN`, paste the token, and enable notebook access. The
cell below picks it up automatically and falls back to an interactive prompt.

If you would rather not deal with gating, set `MODEL` to the Qwen line in the
config cell. It answers the same question; Llama is preferred only because its
GQA geometry matches the architecture discussion in the paper.


In [3]:
import os

def ensure_hf_auth():
    if not MODEL.startswith('meta-llama/'):
        print('Ungated model selected; no token needed.')
        return
    if os.environ.get('HF_TOKEN'):
        print('Using HF_TOKEN from the environment.')
        return
    try:  # Colab secret (key icon in the sidebar)
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
        print('Using HF_TOKEN from Colab secrets.')
        return
    except Exception:
        pass
    from huggingface_hub import notebook_login   # interactive fallback
    notebook_login()

ensure_hf_auth()


Using HF_TOKEN from Colab secrets.


## Setup

Two measurement rules carried over from the paper, both easy to get wrong:

1. **Cold TTFT is a prefill-only forward pass**, never a `generate()` total.
   Comparing a cold prefill+decode against a warm first token inflates the apparent
   speedup by roughly 2.5x.
2. **Every timing is bracketed by `torch.cuda.synchronize()`**, or CUDA's
   asynchronous kernel launches make the GPU look arbitrarily fast.


In [4]:
import torch, time, json, gc, os, tempfile, statistics as st, contextlib
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), 'Switch the runtime to GPU'
device, dtype = 'cuda', torch.float16
cap = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}  compute capability {cap[0]}.{cap[1]}')
print('Flash SDPA requires sm80+:', 'available' if cap[0] >= 8 else 'NOT available, using memory-efficient')

try:
    tok = AutoTokenizer.from_pretrained(MODEL, token=os.environ.get('HF_TOKEN'))
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, torch_dtype=dtype, attn_implementation='sdpa',
        token=os.environ.get('HF_TOKEN')).to(device).eval()
except Exception as e:
    raise SystemExit(
        f'Could not load {MODEL}: {type(e).__name__}: {str(e)[:200]}\n\n'
        'If this mentions gating or a 401/403, the license has not been accepted on '
        'this HuggingFace account, or HF_TOKEN is missing. Accept it at '
        'https://huggingface.co/' + MODEL + ' , or switch MODEL to the Qwen line.')

cfg = model.config
L = cfg.num_hidden_layers
Hq = cfg.num_attention_heads
Hkv = getattr(cfg, 'num_key_value_heads', Hq)
d = cfg.hidden_size // Hq
P = sum(p.numel() for p in model.parameters())
kv_per_token = 2 * L * Hkv * d * 2      # 2 tensors x FP16

def attention_ctx():
    """Prefer a tiled SDPA backend so attention never materializes N x N.

    MATH stays in the list as a last resort. Excluding it turns any kernel
    rejection into 'RuntimeError: No available kernel'; the memory-efficient
    kernel declines whenever the attention mask's last dimension is unaligned,
    which happens as soon as a sequence length is not a multiple of 8.
    """
    if not FORCE_MEM_EFFICIENT_ATTENTION:
        return contextlib.nullcontext()
    try:
        from torch.nn.attention import sdpa_kernel, SDPBackend
        backends = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH]
        if cap[0] >= 8:
            backends.insert(0, SDPBackend.FLASH_ATTENTION)
        return sdpa_kernel(backends)          # ordered by preference
    except Exception:
        return torch.backends.cuda.sdp_kernel(
            enable_flash=cap[0] >= 8, enable_mem_efficient=True, enable_math=True)

print(f'{MODEL}: {L} layers, {Hq} Q-heads, {Hkv} KV-heads, head_dim={d}')
print(f'{P/1e9:.2f}B params, {kv_per_token/1024:.0f} KB KV per token')


GPU: Tesla T4  compute capability 7.5
Flash SDPA requires sm80+: NOT available, using memory-efficient


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

meta-llama/Llama-3.2-1B: 16 layers, 32 Q-heads, 8 KV-heads, head_dim=64
1.24B params, 32 KB KV per token


## Predicted crossover

Generated here from the same expressions as the paper rather than copied, so the
prediction is reproducible from this notebook alone: prefill costs
`(2PN + 4*L*Hq*d*N^2) / (F*eta)` and restore moves `2*L*Hkv*d*b*N` bytes at the
tier's latency and bandwidth.


In [5]:
T4_PEAK, T4_MFU = 65e12, 0.30      # dense FP16 FLOP/s; achievable prefill utilization
NVME_LAT, NVME_BW = 0.010, 2e9

def prefill_s(n):
    return (2 * P * n + 4 * L * Hq * d * n * n) / (T4_PEAK * T4_MFU)

def restore_s(n):
    return NVME_LAT + kv_per_token * n / NVME_BW

n_star = next(n for n in range(1, 200000) if prefill_s(n) > restore_s(n))
print(f'Predicted crossover N* = {n_star:,} tokens (T4, NVMe)')
lo, hi = min(CONTEXT_LENGTHS), max(CONTEXT_LENGTHS)
print(f'Sweep spans {lo}-{hi} tokens = {lo/n_star:.1f}x to {hi/n_star:.1f}x N*')


Predicted crossover N* = 91 tokens (T4, NVMe)
Sweep spans 512-16384 tokens = 5.6x to 180.0x N*


## Measurement

For each context length: prefill cold (timed), serialize the KV cache to disk,
reload it, resume from the restored cache (timed), and verify the resumed next-token
prediction matches the cold path.


In [6]:
TMP = tempfile.mkdtemp()

def make_prompt(n_tokens):
    """Return exactly n_tokens ids.

    Repeats token ids rather than the source string. Tokenizing a repeated
    string merges tokens across the seams, so asking for 2048 tokens that way
    yields something like 2002: an unaligned length that makes the tiled
    attention kernel decline. Powers of two keep the mask aligned.
    """
    base = ('The tiered cache stores attention keys and values across a memory '
            'hierarchy so a resumed conversation need not be recomputed. ')
    unit = tok(base, add_special_tokens=False, return_tensors='pt').input_ids
    reps = -(-n_tokens // unit.shape[1]) + 1
    body = unit.repeat(1, reps)

    if tok.bos_token_id is not None:
        bos = torch.tensor([[tok.bos_token_id]], dtype=body.dtype)
        ids = torch.cat([bos, body], dim=1)[:, :n_tokens]
    else:
        ids = body[:, :n_tokens]

    assert ids.shape[1] == n_tokens, f'wanted {n_tokens}, built {ids.shape[1]}'
    return ids.to(device)


def save_kv(past, path):
    flat = {}
    if hasattr(past, 'key_cache'):
        pairs = list(zip(past.key_cache, past.value_cache))
    else:
        pairs = list(past)
    for i, (k, v) in enumerate(pairs):
        flat[str(i) + '_k'] = k.cpu()
        flat[str(i) + '_v'] = v.cpu()
    torch.save(flat, path)
    return os.path.getsize(path)

def load_kv(path, n_layers):
    flat = torch.load(path)
    keys = [flat[str(i) + '_k'].to(device) for i in range(n_layers)]
    vals = [flat[str(i) + '_v'].to(device) for i in range(n_layers)]
    legacy = tuple(zip(keys, vals))
    try:
        from transformers import DynamicCache
        return DynamicCache.from_legacy_cache(legacy)
    except Exception:
        return legacy


In [7]:
results = []

for n in CONTEXT_LENGTHS:
    try:
        ids = make_prompt(n)
        actual = ids.shape[1]
        cold_ms, warm_ms, sizes, identical, logit_deltas = [], [], [], [], []

        for _ in range(TRIALS):
            with attention_ctx():
                torch.cuda.synchronize(); t0 = time.perf_counter()
                with torch.no_grad():
                    out = model(ids, use_cache=True)
                torch.cuda.synchronize()
                cold_ms.append((time.perf_counter() - t0) * 1000)
                cold_logits = out.logits[:, -1, :].float().cpu()

                path = os.path.join(TMP, 'kv.pt')
                sizes.append(save_kv(out.past_key_values, path))
                del out; gc.collect(); torch.cuda.empty_cache()

                torch.cuda.synchronize(); t1 = time.perf_counter()
                past = load_kv(path, L)
                with torch.no_grad():
                    warm = model(ids[:, -1:], past_key_values=past, use_cache=True)
                torch.cuda.synchronize()
                warm_ms.append((time.perf_counter() - t1) * 1000)
                warm_logits = warm.logits[:, -1, :].float().cpu()

            # Fidelity: argmax agreement is a coarse proxy that a near-tie can flip
            # even when the restore is bit-faithful. The logit delta says whether
            # the restored state is actually the same state.
            identical.append(int(warm_logits.argmax(-1)) == int(cold_logits.argmax(-1)))
            logit_deltas.append(float((warm_logits - cold_logits).abs().max()))
            del warm, past; gc.collect(); torch.cuda.empty_cache()

        row = {
            'tokens': actual,
            'cold_ttft_ms': st.median(cold_ms),
            'warm_path_ms': st.median(warm_ms),
            'speedup': st.median(cold_ms) / st.median(warm_ms),
            'kv_mb': st.median(sizes) / 1e6,
            'predicted_prefill_ms': prefill_s(actual) * 1000,
            'predicted_restore_ms': restore_s(actual) * 1000,
            'next_token_identical': str(sum(identical)) + '/' + str(len(identical)),
            'max_logit_delta': max(logit_deltas),
            'trials': TRIALS,
        }
        results.append(row)
        print('{:>6} tok | cold {:>8.1f}ms | warm {:>7.1f}ms | {:>6.2f}x | KV {:>6.1f}MB'
              ' | id {} | max dlogit {:.2e}'.format(
            actual, row['cold_ttft_ms'], row['warm_path_ms'], row['speedup'],
            row['kv_mb'], row['next_token_identical'], row['max_logit_delta']))

    except torch.cuda.OutOfMemoryError as e:
        print(f'{n} tok | OUT OF MEMORY, stopping sweep here. Results so far are kept.')
        print(f'  {type(e).__name__}: {str(e)[:120]}')
        gc.collect(); torch.cuda.empty_cache()
        break

    # Persist after every length so a later failure cannot lose earlier work.
    with open('context_sweep_results_partial.json', 'w') as f:
        json.dump(results, f, indent=2)


   512 tok | cold     89.0ms | warm    45.6ms |   1.95x | KV   16.8MB | id 5/5 | max dlogit 7.91e+00
  2048 tok | cold    708.1ms | warm    77.1ms |   9.18x | KV   67.1MB | id 5/5 | max dlogit 7.58e+00
  4096 tok | cold   2517.2ms | warm   159.5ms |  15.78x | KV  134.2MB | id 5/5 | max dlogit 1.64e+01
8192 tok | OUT OF MEMORY, stopping sweep here. Results so far are kept.
  OutOfMemoryError: CUDA out of memory. Tried to allocate 8.00 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.45 GiB is free. Inclu


## Results


In [8]:
summary = {
    'model': MODEL,
    'gpu': torch.cuda.get_device_name(0),
    'geometry': {'layers': L, 'q_heads': Hq, 'kv_heads': Hkv, 'head_dim': d,
                 'params': P, 'kv_bytes_per_token': kv_per_token},
    'predicted_n_star': n_star,
    'mfu_assumed': T4_MFU,
    'rows': results,
}
with open('context_sweep_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('{:>8}{:>12}{:>12}{:>10}'.format('tokens', 'measured', 'predicted', 'speedup'))
for r in results:
    print('{:>8}{:>10.0f}ms{:>10.0f}ms{:>9.2f}x'.format(
        r['tokens'], r['cold_ttft_ms'], r['predicted_prefill_ms'], r['speedup']))
print()
print('Saved context_sweep_results.json -- download this file.')


  tokens    measured   predicted   speedup
     512        89ms        67ms     1.95x
    2048       708ms       288ms     9.18x
    4096      2517ms       632ms    15.78x

Saved context_sweep_results.json -- download this file.


In [9]:
# Download the results file. In Colab this opens a browser download; elsewhere
# it is a no-op and the file is simply left on disk.
try:
    from google.colab import files
    files.download('context_sweep_results.json')
except Exception as e:
    print('Not running in Colab, or download blocked:', e)
    import os
    print('File is at:', os.path.abspath('context_sweep_results.json'))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### What to look for

- **Speedup should rise monotonically** with context length and stay above 1x across
  the whole sweep, since every point sits far above N*.
- **`max dlogit`** is the real fidelity check. Values around 1e-3 or smaller mean the
  restored cache reproduces the cold-path state and any `id` mismatch is a near-tie
  argmax flip, not a broken restore. Values of order 1 would mean the restore is wrong.
  `id` is reported alongside it because the paper's earlier TinyLlama runs used that
  coarser check.
- **Measured vs predicted cold TTFT.** A roughly constant ratio means the assumed MFU
  needs recalibrating for this GPU. A ratio that *grows* with context length means the
  quadratic attention term is running unoptimized, which is what happens when SDPA falls
  back to the math backend. That is why this notebook forces a tiled backend.
